In [29]:
import pandas as pd
import sys
import os
import importlib

sys.path.append(os.path.abspath("../preprocessing"))

import preprocess_churn
importlib.reload(preprocess_churn)

from preprocess_churn import load_and_prepare_data

In [30]:
data = load_and_prepare_data()

df = data["df"]
X = data["X"]
y = data["y"]
X_train = data["X_train"]
X_test = data["X_test"]
y_train = data["y_train"]
y_test = data["y_test"]
preprocessor = data["preprocessor"]

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\ny_train distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\ny_test distribution:")
print(y_test.value_counts(normalize=True) * 100)

X_train shape: (4504, 27)
X_test shape: (1126, 27)

y_train distribution:
churn
0    83.170515
1    16.829485
Name: proportion, dtype: float64

y_test distribution:
churn
0    83.12611
1    16.87389
Name: proportion, dtype: float64


In [31]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

In [32]:
def evaluate_model(model_name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    results = {
        "model": model_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1_score": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_pred_proba)
    }

    print(f"{model_name} Results")
    print("Accuracy:", results["accuracy"])
    print("Precision:", results["precision"])
    print("Recall:", results["recall"])
    print("F1-score:", results["f1_score"])
    print("ROC-AUC:", results["roc_auc"])

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    return results

In [33]:
log_reg_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
    ]
)

log_reg_model.fit(X_train, y_train)

log_reg_results = evaluate_model(
    "Logistic Regression",
    log_reg_model,
    X_test,
    y_test
)

Logistic Regression Results
Accuracy: 0.8028419182948491
Precision: 0.4553072625698324
Recall: 0.8578947368421053
F1-score: 0.5948905109489051
ROC-AUC: 0.8937471884840306

Confusion Matrix:
[[741 195]
 [ 27 163]]

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.79      0.87       936
           1       0.46      0.86      0.59       190

    accuracy                           0.80      1126
   macro avg       0.71      0.82      0.73      1126
weighted avg       0.88      0.80      0.82      1126



In [34]:
dt_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", DecisionTreeClassifier(
            random_state=42,
            class_weight="balanced"
        ))
    ]
)

dt_model.fit(X_train, y_train)

dt_results = evaluate_model(
    "Decision Tree",
    dt_model,
    X_test,
    y_test
)

Decision Tree Results
Accuracy: 0.9724689165186501
Precision: 0.8955223880597015
Recall: 0.9473684210526315
F1-score: 0.9207161125319693
ROC-AUC: 0.9624662618083669

Confusion Matrix:
[[915  21]
 [ 10 180]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.98      0.98       936
           1       0.90      0.95      0.92       190

    accuracy                           0.97      1126
   macro avg       0.94      0.96      0.95      1126
weighted avg       0.97      0.97      0.97      1126



In [35]:
rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        ))
    ]
)

rf_model.fit(X_train, y_train)

rf_results = evaluate_model(
    "Random Forest V1",
    rf_model,
    X_test,
    y_test
)

Random Forest V1 Results
Accuracy: 0.9751332149200711
Precision: 0.9655172413793104
Recall: 0.8842105263157894
F1-score: 0.9230769230769231
ROC-AUC: 0.9982512370670265

Confusion Matrix:
[[930   6]
 [ 22 168]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.99      0.99       936
           1       0.97      0.88      0.92       190

    accuracy                           0.98      1126
   macro avg       0.97      0.94      0.95      1126
weighted avg       0.97      0.98      0.97      1126



In [36]:
data_v2 = load_and_prepare_data(
    drop_additional_cols=["is_new_customer", "has_complaint"]
)

X_train_v2 = data_v2["X_train"]
X_test_v2 = data_v2["X_test"]
y_train_v2 = data_v2["y_train"]
y_test_v2 = data_v2["y_test"]
preprocessor_v2 = data_v2["preprocessor"]

rf_model_v2 = Pipeline(
    steps=[
        ("preprocessor", preprocessor_v2),
        ("model", RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        ))
    ]
)

rf_model_v2.fit(X_train_v2, y_train_v2)

rf_v2_results = evaluate_model(
    "Random Forest V2",
    rf_model_v2,
    X_test_v2,
    y_test_v2
)

Random Forest V2 Results
Accuracy: 0.9733570159857904
Precision: 0.9761904761904762
Recall: 0.8631578947368421
F1-score: 0.9162011173184358
ROC-AUC: 0.9984058704453441

Confusion Matrix:
[[932   4]
 [ 26 164]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      1.00      0.98       936
           1       0.98      0.86      0.92       190

    accuracy                           0.97      1126
   macro avg       0.97      0.93      0.95      1126
weighted avg       0.97      0.97      0.97      1126



In [37]:
model_results = pd.DataFrame([
    log_reg_results,
    dt_results,
    rf_results,
    rf_v2_results
])

model_results

,model,accuracy,precision,recall,f1_score,roc_auc
0,Logistic Regression,0.802842,0.455307,0.857895,0.594891,0.893747
1,Decision Tree,0.972469,0.895522,0.947368,0.920716,0.962466
2,Random Forest V1,0.975133,0.965517,0.884211,0.923077,0.998251
3,Random Forest V2,0.973357,0.976190,0.863158,0.916201,0.998406


## Training Summary

Four supervised learning models were trained for churn prediction: Logistic Regression, Decision Tree, Random Forest V1, and Random Forest V2.

Logistic Regression was used as a baseline model because it is simple and interpretable.

Decision Tree achieved the highest recall, meaning it captured the largest share of actual churned customers.

Random Forest V1 achieved the best overall balance, with the highest F1-score and very high ROC-AUC.

Random Forest V2 was trained after removing potentially redundant engineered features, including `is_new_customer` and `has_complaint`. Its performance remained strong, which suggests that the model is not fully dependent on those engineered features.

The next step is to evaluate and compare model performance in more detail.

In [38]:
import os

os.makedirs("../../06_models", exist_ok=True)
os.makedirs("../../05_outputs/model_results", exist_ok=True)

In [39]:
model_results.to_csv(
    "../../05_outputs/model_results/model_comparison.csv",
    index=False
)

In [40]:
import joblib

joblib.dump(
    rf_model,
    "../../06_models/random_forest_churn_model.pkl"
)

['../../06_models/random_forest_churn_model.pkl']

In [41]:
print(os.path.exists("../../05_outputs/model_results/model_comparison.csv"))
print(os.path.exists("../../06_models/random_forest_churn_model.pkl"))

True
True
